In [1]:
# Init input 
import torch
import cutlass
from cutlass import cute
from cutlass.cute.runtime import from_dlpack
import cutlass.utils as utils

M, N, K = 1024, 1024, 256
A  = from_dlpack(torch.randn(M, K))
B  = from_dlpack(torch.randn(N, K).permute(1, 0))
C  = from_dlpack(torch.zeros(M, N))

In [2]:
print(utils.LayoutEnum.from_tensor(A))
print(utils.LayoutEnum.from_tensor(B))
print(utils.LayoutEnum.from_tensor(C))

LayoutEnum.ROW_MAJOR
LayoutEnum.COL_MAJOR
LayoutEnum.ROW_MAJOR


# Make A,B load layout 

Gemm pipeline
GMEM → (async copy) → SMEM → (ldmatrix) → RMEM → (tensor core MMA) → RMEM(acc) → SMEM → GMEM

## SMEM Layout for A and B 




In [ ]:



# MMA instruction shape for Ampere (SM80)
# m16n8k16: one warp computes a 16×8 output tile consuming 16 K elements

class GEMMV1():
    def __init__(self, tiler=(128, 128, 32)):
        # MMA instruction shape for Ampere (SM80)
        # m16n8k16: one warp computes a 16×8 output tile consuming 16 K elements
        self.mma_shape= (16, 8, 16)
        self.atom_layout= (2,2,1) 
        self.num_threads = self.atom_layout[0] * self.atom_layout[1] * self.atom_layout[2] * 32 # 32 threads per warp
        self.tiler = tiler
        self.bM, self.bN, self.bK = self.tiler
        self.num_stages = 1 
        self.AB_dtype = cutlass.Float16
        self.ACC_dtype = cutlass.Float32
        self.copy_bits = 128 # copy 128 bits or 8 elements at a time for half precision

    @cute.jit
    def __call__(self, A:cute.Tensor, B:cute.Tensor, C:cute.Tensor):



    @cute.kernel
    def gemm_kernel(self, A:cute.Tensor, B:cute.Tensor, C:cute.Tensor):
        pass

    @cute.jit
    def make_sAB_layout(self, A:cute.Tensor, B:cute.Tensor, C:cute.Tensor):
        cute.make_layout()

    @cute.jit 
    def make_gAB_layout(self, A:cute.Tensor, B:cute.Tensor, C:cute.Tensor):
        cute.make_layout()

        
    @cute.jit
    def make_t2s_AB_laytout(self, gA_layout, gB_layout):
        pass
        




        

In [ ]:
make_loadAB_layout(A, B, C)


(1024,1024):(1024,1)
(1024,256):(256,1)
(1024,256):(256,1)


# Construct A,B Copy Atom

In [ ]:
# Make A, B Layout 

In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="ear", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores


In [ ]:
from transformers import Sam3VideoModel, Sam3VideoProcessor
from accelerate import Accelerator
import torch

device = Accelerator().device
model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=torch.bfloat16)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

# Load video frames
from transformers.video_utils import load_video
video_url = "https://huggingface.co/datasets/hf-internal-testing/sam2-fixtures/resolve/main/bedroom.mp4"
video_frames, _ = load_video(video_url)

# Initialize video inference session
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=device,
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=torch.bfloat16,
)

# Add text prompt to detect and track objects
text = "person"
inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text=text,
)

# Process all frames in the video
outputs_per_frame = {}
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session, max_frame_num_to_track=50
):
    processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed_outputs

print(f"Processed {len(outputs_per_frame)} frames")

# Access results for a specific frame
frame_0_outputs = outputs_per_frame[0]
print(f"Detected {len(frame_0_outputs['object_ids'])} objects")
print(f"Object IDs: {frame_0_outputs['object_ids'].tolist()}")
print(f"Scores: {frame_0_outputs['scores'].tolist()}")
print(f"Boxes shape (XYXY format, absolute coordinates): {frame_0_outputs['boxes'].shape}")
print(f"Masks shape: {frame_0_outputs['masks'].shape}")
